# 1 ) Feature Engineering
Objective

* The objective of feature engineering is to create meaningful and predictive features from machine sensor readings, maintenance history, and error logs that improve machine failure prediction performance.

## 1.1 import required libraries

In [2]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## 1.3 Load Merged Dataset

In [4]:
# Load merged dataset

merged_df = pd.read_csv(
    "merged_dataset.csv"
)

# Convert datetime column

merged_df['datetime'] = pd.to_datetime(
    merged_df['datetime']
)

# Check shape

print(merged_df.shape)

(876100, 18)


 ## 1.3 Total Error Count
 * Machines generating frequent errors are more likely to experience failures.


In [9]:
# Calculate total machine errors

error_cols = [
    'errorID_error1',
    'errorID_error2',
    'errorID_error3',
    'errorID_error4',
    'errorID_error5'
]

merged_df['total_error_count'] = merged_df[error_cols].sum(axis=1)

## 1.4 Total Maintenance Count
* Maintenance frequency provides insights into machine health and service history

In [10]:
# Calculate total maintenance activities

maint_cols = [
    'comp_comp1',
    'comp_comp2',
    'comp_comp3',
    'comp_comp4'
]

merged_df['total_maintenance_count'] = merged_df[maint_cols].sum(axis=1)

## 1.5 Machine Age Category
* Older machines generally have a higher risk of failure.

In [38]:
merged_df['age_category'] = pd.cut(
    merged_df['age'],
    bins=[0,5,10,25],
    labels=['New','Mid','Old'],
    include_lowest=True
)

In [20]:
# Sort dataset before creating rolling features

merged_df = merged_df.sort_values(
    ['machineID', 'datetime']
)

## 1.6 Voltage Fluctuation Feature
* Voltage instability can indicate abnormal machine behavior.

In [14]:
# Calculate rolling voltage standard deviation

merged_df['voltage_std_24h'] = (
    merged_df
    .groupby('machineID')['volt']
    .rolling(window=24, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
)

## 1.6 Pressure Fluctuation Feature
* Pressure fluctuations often indicate machine degradation.

In [15]:
# Calculate rolling pressure standard deviation

merged_df['pressure_std_24h'] = (
    merged_df
    .groupby('machineID')['pressure']
    .rolling(window=24, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
)

## 1.7 Vibration Fluctuation Feature
* Abnormal vibration is one of the strongest indicators of machine failure.

In [16]:
# Calculate rolling vibration standard deviation

merged_df['vibration_std_24h'] = (
    merged_df
    .groupby('machineID')['vibration']
    .rolling(window=24, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
)

## 1.8 Rolling Voltage Mean
* Captures recent voltage behavior over the previous 24 hours.

In [17]:
# Calculate rolling voltage mean

merged_df['rolling_voltage_mean'] = (
    merged_df
    .groupby('machineID')['volt']
    .rolling(window=24, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

## 1.9 Rolling Pressure Mean
* Provides recent pressure trends for each machine.

In [18]:
# Calculate rolling pressure mean

merged_df['rolling_pressure_mean'] = (
    merged_df
    .groupby('machineID')['pressure']
    .rolling(window=24, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

## 1.10 Rolling Vibration Mean
* Captures recent vibration behavior and machine condition.

In [19]:
# Calculate rolling vibration mean

merged_df['rolling_vibration_mean'] = (
    merged_df
    .groupby('machineID')['vibration']
    .rolling(window=24, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

## 1.11 Machine Health Index
* Combines multiple sensor readings into a single machine condition indicator.

In [21]:
# Create machine health index

merged_df['health_index'] = (
    merged_df['volt'] +
    merged_df['pressure'] +
    merged_df['vibration'] +
    merged_df['rotate']
) / 4

## 1.12 Voltage-Vibration Ratio
* Captures the interaction between electrical and mechanical behavior.

In [22]:
# Calculate voltage-vibration ratio

merged_df['volt_vibration_ratio'] = (
    merged_df['volt'] /
    (merged_df['vibration'] + 1e-5)
)

## 1.13 Error-Maintenance Ratio
* Measures whether maintenance activity is sufficient relative to machine issues.

In [23]:
# Calculate error-maintenance ratio

merged_df['error_maintenance_ratio'] = (
    merged_df['total_error_count'] /
    (merged_df['total_maintenance_count'] + 1)
)

## 1.14 Feature Engineering Validation
* To verify that all engineered features were created successfully.


In [24]:
# Check dataset shape

print("Dataset Shape:", merged_df.shape)

# Display sample records

merged_df.head()

Dataset Shape: (876100, 30)


,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID_error1,errorID_error2,...,voltage_std_24h,pressure_std_24h,total_maintenance_count,vibration_std_24h,rolling_voltage_mean,rolling_pressure_mean,rolling_vibration_mean,health_index,volt_vibration_ratio,error_maintenance_ratio
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,0.0,0.0,...,NaN,NaN,0.0,NaN,176.217853,113.077935,45.087686,188.221888,3.908336,0.0
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,0.0,0.0,...,9.431836,12.457390,0.0,1.183494,169.548538,104.269230,44.250829,176.125303,3.751769,0.0
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,0.0,0.0,...,6.721032,18.934956,0.0,5.874970,170.028993,94.592122,40.893502,201.939120,5.002798,0.0
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,0.0,0.0,...,6.665324,17.109194,0.0,4.798255,168.137453,98.256232,40.950662,164.745718,3.950737,0.0
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,0.0,0.0,...,7.448844,16.021893,0.0,7.875828,166.031967,100.982315,37.958632,182.716013,6.064135,0.0


# 2 ) Creation of Synthetic Features
Objective

* In addition to the original sensor, maintenance, and error-related features, four synthetic features were created using domain knowledge from smart manufacturing and predictive maintenance.

* These features are not directly available in the original dataset but are derived from existing machine measurements to better represent machine workload, energy usage, electrical behavior, and operational stress.

## 2.1 Synthetic Features Created
1. Production Load
2. Current Consumption
3. Energy Consumption
4. Machine Stress Index

## 2.2 Mathematical Representation
1. Production Load

* Measures machine utilization relative to the maximum observed rotational speed.

* Production Load=(Rotate/Max(Rotate))*100


2. Current Consumption

* Estimates electrical current demand using available voltage measurements.

* Current=Voltage/10

3. Energy Consumption

* Estimates machine energy usage based on voltage and current.

* Energy Consumption=Voltage*Current


4. Machine Stress Index

* Represents overall mechanical stress by combining pressure and vibration.

* Machine Stress Index=Pressure*Vibration



	​


	​



## 2.3 Production Load
* Production Load estimates how heavily a machine is being utilized based on its rotational speed. Machines operating at higher speeds are generally subjected to greater workload and mechanical stress.

In [25]:
# Calculate production load percentage

merged_df['production_load'] = (
    merged_df['rotate'] /
    merged_df['rotate'].max()
) * 100

## 2.4 Current Consumption
* Current Consumption is a synthetic electrical feature derived from voltage measurements. It provides an estimate of machine power demand and operational intensity.

In [26]:
# Estimate current consumption

merged_df['current'] = (
    merged_df['volt'] / 10
)

## 2.5 Energy Consumption
* Energy Consumption estimates the energy utilized by a machine during operation. Machines consuming higher energy may indicate heavier workloads or inefficient operating conditions

In [27]:
# Calculate energy consumption

merged_df['energy_consumption'] = (
    merged_df['volt'] *
    merged_df['current']
)

## Machine Stress Index
* Machine Stress Index combines pressure and vibration measurements into a single indicator. High pressure together with excessive vibration often reflects abnormal machine operating conditions.

In [28]:
# Calculate machine stress index

merged_df['machine_stress_index'] = (
    merged_df['pressure'] *
    merged_df['vibration']
)

# 3 ) Feature Engineering Validation
Objective

* To verify the newly created features, inspect dataset quality, and ensure the dataset is ready for feature selection and modeling.

## 3.1 Display All Columns
* To verify that all original and engineered features have been successfully created.

In [29]:
# Display all column names

for col in merged_df.columns:
    print(col)

datetime
machineID
volt
rotate
pressure
vibration
model
age
errorID_error1
errorID_error2
errorID_error3
errorID_error4
errorID_error5
comp_comp1
comp_comp2
comp_comp3
comp_comp4
failure_flag
total_error_count
age_category
voltage_std_24h
pressure_std_24h
total_maintenance_count
vibration_std_24h
rolling_voltage_mean
rolling_pressure_mean
rolling_vibration_mean
health_index
volt_vibration_ratio
error_maintenance_ratio
production_load
current
energy_consumption
machine_stress_index


## 3.2 Total Number of Features
* To understand the final feature count after feature engineering.

In [30]:
# Count total columns

print(
    "Total Features:",
    len(merged_df.columns)
)

Total Features: 34


## 3.3 Dataset Information
* To verify data types and memory usage.

In [31]:
# Dataset information

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 876100 entries, 0 to 876099
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   datetime                 876100 non-null  datetime64[ns]
 1   machineID                876100 non-null  int64         
 2   volt                     876100 non-null  float64       
 3   rotate                   876100 non-null  float64       
 4   pressure                 876100 non-null  float64       
 5   vibration                876100 non-null  float64       
 6   model                    876100 non-null  object        
 7   age                      876100 non-null  int64         
 8   errorID_error1           876100 non-null  float64       
 9   errorID_error2           876100 non-null  float64       
 10  errorID_error3           876100 non-null  float64       
 11  errorID_error4           876100 non-null  float64       
 12  errorID_error5  

## 3.4 Check Sample Records

In [32]:
# Display first 5 records

merged_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID_error1,errorID_error2,...,rolling_voltage_mean,rolling_pressure_mean,rolling_vibration_mean,health_index,volt_vibration_ratio,error_maintenance_ratio,production_load,current,energy_consumption,machine_stress_index
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,0.0,0.0,...,176.217853,113.077935,45.087686,188.221888,3.908336,0.0,60.214596,17.621785,3105.273172,5098.422421
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,0.0,0.0,...,169.548538,104.269230,44.250829,176.125303,3.751769,0.0,57.947529,16.287922,2652.964125,4144.320641
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,0.0,0.0,...,170.028993,94.592122,40.893502,201.939120,5.002798,0.0,75.875382,17.098990,2923.754672,2571.544848
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,0.0,0.0,...,168.137453,98.256232,40.950662,164.745718,3.950737,0.0,49.804156,16.246283,2639.417219,4492.535078
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,0.0,0.0,...,166.031967,100.982315,37.958632,182.716013,6.064135,0.0,62.642263,15.761002,2484.091878,2907.991161


# 4 ) Check Missing Values

In [33]:
# Missing value check

print(
    "Missing Values:",
    merged_df.isnull().sum().sum()
)

Missing Values: 9061


In [34]:
# Duplicate row check

print(
    "Duplicate Rows:",
    merged_df.duplicated().sum()
)

Duplicate Rows: 0


In [35]:
# Duplicate machineID-datetime check

print(
    "Duplicate Keys:",
    merged_df[
        ['machineID','datetime']
    ].duplicated().sum()
)

Duplicate Keys: 0


In [36]:
# Display engineered features

engineered_features = [

    'total_error_count',
    'total_maintenance_count',
    'voltage_std_24h',
    'pressure_std_24h',
    'vibration_std_24h',
    'rolling_voltage_mean',
    'rolling_pressure_mean',
    'rolling_vibration_mean',
    'health_index',
    'volt_vibration_ratio',
    'error_maintenance_ratio',
    'production_load',
    'current',
    'energy_consumption',
    'machine_stress_index'

]

merged_df[
    engineered_features
].head()

,total_error_count,total_maintenance_count,voltage_std_24h,pressure_std_24h,vibration_std_24h,rolling_voltage_mean,rolling_pressure_mean,rolling_vibration_mean,health_index,volt_vibration_ratio,error_maintenance_ratio,production_load,current,energy_consumption,machine_stress_index
0,0.0,0.0,NaN,NaN,NaN,176.217853,113.077935,45.087686,188.221888,3.908336,0.0,60.214596,17.621785,3105.273172,5098.422421
1,0.0,0.0,9.431836,12.457390,1.183494,169.548538,104.269230,44.250829,176.125303,3.751769,0.0,57.947529,16.287922,2652.964125,4144.320641
2,0.0,0.0,6.721032,18.934956,5.874970,170.028993,94.592122,40.893502,201.939120,5.002798,0.0,75.875382,17.098990,2923.754672,2571.544848
3,0.0,0.0,6.665324,17.109194,4.798255,168.137453,98.256232,40.950662,164.745718,3.950737,0.0,49.804156,16.246283,2639.417219,4492.535078
4,0.0,0.0,7.448844,16.021893,7.875828,166.031967,100.982315,37.958632,182.716013,6.064135,0.0,62.642263,15.761002,2484.091878,2907.991161


In [37]:
# Column-wise missing values

missing_df = pd.DataFrame({
    'Column': merged_df.columns,
    'Missing Values': merged_df.isnull().sum()
})

missing_df = missing_df[
    missing_df['Missing Values'] > 0
]

missing_df.sort_values(
    by='Missing Values',
    ascending=False
)

,Column,Missing Values
age_category,age_category,8761
voltage_std_24h,voltage_std_24h,100
pressure_std_24h,pressure_std_24h,100
vibration_std_24h,vibration_std_24h,100


# 5 ) Handling missing values

In [40]:
# Create age category feature

merged_df['age_category'] = pd.cut(
    merged_df['age'],
    bins=[0,5,10,25],
    labels=['New','Mid','Old'],
    include_lowest=True
)

In [41]:
merged_df['age_category'].isnull().sum()

np.int64(0)

In [42]:
#  Fix 2: Voltage Standard Deviation
# Replace missing values

merged_df['voltage_std_24h'] = (
    merged_df['voltage_std_24h']
    .fillna(0)
)

In [43]:
#  validation
print(
    merged_df['voltage_std_24h']
    .isnull()
    .sum()
)

0


In [46]:
#  Fix 3: Pressure Standard Deviation
# Replace missing values

merged_df['pressure_std_24h'] = (
    merged_df['pressure_std_24h']
    .fillna(0)
)

In [47]:
# validation
print(
    merged_df['pressure_std_24h']
    .isnull()
    .sum()
)

0


In [44]:
#  Fix 4: Vibration Standard Deviation
# Replace missing values

merged_df['vibration_std_24h'] = (
    merged_df['vibration_std_24h']
    .fillna(0)
)

In [45]:
#  validation
print(
    merged_df['vibration_std_24h']
    .isnull()
    .sum()
)


0


## 5.1 Final validation

In [48]:
# Check total missing values

print(
    "Total Missing Values:",
    merged_df.isnull().sum().sum()
)

Total Missing Values: 0


# 6 ) Save Feature Engineered Dataset
* Save the final feature-engineered dataset for feature selection, model training, and future analysis.

In [49]:
# Save feature engineered dataset

merged_df.to_csv(
    "feature_engineered_dataset.csv",
    index=False
)

print(
    "Feature engineered dataset saved successfully."
)

Feature engineered dataset saved successfully.


## 6.1 Verify Dataset Saved

In [50]:
# Check dataset dimensions

print(
    "Dataset Shape:",
    merged_df.shape
)

Dataset Shape: (876100, 34)


## 6.2 Final Dataset Quality Check

In [51]:
# Final validation before modeling

print(
    "Duplicate Rows:",
    merged_df.duplicated().sum()
)

print(
    "Duplicate Keys:",
    merged_df[
        ['machineID','datetime']
    ].duplicated().sum()
)

print(
    "Missing Values:",
    merged_df.isnull().sum().sum()
)

Duplicate Rows: 0
Duplicate Keys: 0
Missing Values: 0


# 7 ) Observation
* The feature-engineered dataset was successfully validated and exported. All engineered features were generated correctly, missing values were handled, and no duplicate observations were found. The dataset is now ready for post-feature-engineering analysis, feature selection, class balancing, and predictive maintenance model development.
